In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.0
```
</details>
Dacă ai un set de $n$ biți (sau qubiți), de obicei vei eticheta fiecare bit $0
\rightarrow n-1$. Diferitele programe software și resurse trebuie să aleagă cum ordonează
acești biți atât în memoria calculatorului, cât și atunci când sunt afișați pe ecran.

## Convențiile Qiskit

Iată cum ordonează SDK-ul Qiskit biții în diferite scenarii.

### Circuit-uri cuantice

Clasa `QuantumCircuit` își stochează qubiții într-o listă
(`QuantumCircuit.qubits`). Indexul unui qubit în această listă definește
eticheta qubitului.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import Qubit

qc = QuantumCircuit(2)
qc.qubits[0]  # qubit "0"

Qubit(QuantumRegister(2, "q"), 0)

<Qubit register=(2, "q"), index=0>

### Circuit diagrams

On a circuit diagram, qubit $0$ is the topmost qubit, and qubit $n-1$ the
bottommost qubit. You can change this with the `reverse_bits` argument of
`QuantumCircuit.draw` (see [Change ordering in
Qiskit](#change-ordering-in-qiskit)).

In [2]:
qc.x(1)
qc.draw()

          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

### Diagrame de Circuit
Într-o diagramă de Circuit, qubitul $0$ este qubitul din partea de sus, iar qubitul $n-1$ este
cel din partea de jos. Poți schimba acest lucru cu argumentul `reverse_bits` al
`QuantumCircuit.draw` (vezi [Schimbarea ordonării în
Qiskit](#change-ordering-in-qiskit)).

In [3]:
from qiskit.primitives import StatevectorSampler as Sampler

qc.measure_all()

job = Sampler().run([qc])
result = job.result()
print(f" > Counts: {result[0].data.meas.get_counts()}")

 > Counts: {'10': 1024}


### Strings

When displaying or interpreting a list of bits (or qubits) as a string, bit
$n-1$ is the leftmost bit, and bit $0$ is the rightmost bit. This is because we
usually write numbers with the most significant digit on the left, and in
Qiskit, bit $n-1$ is interpreted as the most significant bit.

For example, the following cell defines a `Statevector` from a string of
single-qubit states. In this case, qubit $0$ is in state $|+\rangle$, and
qubit $1$ in state $|0\rangle$.

In [4]:
from qiskit.quantum_info import Statevector

sv = Statevector.from_label("0+")
sv.probabilities_dict()

{np.str_('00'): np.float64(0.4999999999999999),
 np.str_('01'): np.float64(0.4999999999999999)}

### Numere întregi
Când interpretăm biții ca un număr, bitul $0$ este bitul cel mai puțin semnificativ, iar
bitul $n-1$ este cel mai semnificativ. Acest lucru este util la programare, deoarece fiecare bit are
valoarea $2^\text{label}$ (eticheta fiind indexul qubitului în
`QuantumCircuit.qubits`). De exemplu, următorul Circuit se termină
cu bitul $0$ fiind `0` și bitul $1$ fiind `1`. Aceasta este interpretată ca numărul întreg
zecimal `2` (măsurat cu probabilitatea `1.0`).

In [5]:
print(sv[1])  # amplitude of state |01>
print(sv[2])  # amplitude of state |10>

(0.7071067811865475+0j)
0j


### Gates

Each gate in Qiskit can interpret a list of qubits in its own way, but
controlled gates usually follow the convention `(control, target)`.

For example, the following cell adds a controlled-X gate where qubit $0$ is
the control and qubit $1$ is the target.

In [6]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.cx(0, 1)
qc.draw()

          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘

### Șiruri de caractere
Când afișăm sau interpretăm o listă de biți (sau qubiți) ca un șir de caractere, bitul
$n-1$ este bitul din stânga, iar bitul $0$ este bitul din dreapta. Aceasta deoarece
de obicei scriem numerele cu cifra cea mai semnificativă în stânga, iar în
Qiskit, bitul $n-1$ este interpretat ca bitul cel mai semnificativ.

De exemplu, următoarea celulă definește un `Statevector` dintr-un șir de
stări cu un singur qubit. În acest caz, qubitul $0$ se află în starea $|+\rangle$, iar
qubitul $1$ în starea $|0\rangle$.

In [7]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.x(0)
qc.draw(reverse_bits=True)

          
q_1: ─────
     ┌───┐
q_0: ┤ X ├
     └───┘

You can use the `reverse_bits` method to return a new circuit with the
qubits' labels reversed (this does not mutate the original circuit).

In [8]:
qc.reverse_bits().draw()

          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

Aceasta poate cauza confuzie uneori la interpretarea unui șir de biți, deoarece
s-ar putea să te aștepți ca bitul din stânga să fie bitul $0$, în timp ce de obicei reprezintă bitul
$n-1$.

### Matrice Statevector
Când se reprezintă un statevector ca o listă de numere complexe (amplitudini),
Qiskit ordonează aceste amplitudini astfel încât amplitudinea de la indexul $x$ reprezintă
starea bazei computaționale $|x\rangle$.